# XGBoost to predict diamond prices

What is a diamond worth? A jeweler has records for 53,940 diamonds: the weight in carats, the quality of the cut, the color, the clarity, the measurements, and the price each one sold for. Build a model that predicts the price of a diamond from its other features, and find out how much XGBoost improves on a single decision tree.

Replace each `?` with the missing code. Answer each question in the cell below it.

## Roadmap

Use this map to follow the code blocks below. Stages A to E happen in Steps 1 and 2. Stage F is Steps 4 through 7, stage G is Step 8, and stage H is Step 9. Step 3 is the side trip to simpler models. Step 10 repeats the pattern on a small dataset you have used before.

XGBoost roadmap

In [ ]:
%pip install xgboost scikit-learn seaborn

### Step 1: Load and Explore the Data

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, confusion_matrix
from xgboost import XGBRegressor, XGBClassifier

# Load the diamonds dataset that comes with seaborn
df = sns.load_dataset('?')

df

In [ ]:
# Check the column types and look for missing values
df.info()

### Step 2: Define predictors and target

Predict `price` from the other columns. The text columns need one-hot encoding. Then split the rows twice: once to hold back the test rows, and again to hold back validation rows from the training rows.

In [ ]:
target = '?'
numeric_features = ['?']
categorical_features = ['?']

X = df[numeric_features + categorical_features]
y = df[target]

# One-Hot Encode the categorical predictors as 1/0 variables
X_encoded = pd.get_dummies(X, columns=categorical_features, drop_first=False, dtype=int)

# Hold back 25% of the rows as test rows
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=?, random_state=42
)

# Hold back 20% of the training rows as validation rows
X_fit, X_val, y_fit, y_val = train_test_split(
    ?, ?, test_size=0.2, random_state=42
)

print("Rows to learn from:", len(X_fit))
print("Validation rows:", len(X_val))
print("Test rows:", len(X_test))

### Step 3: Baseline models

Fit a linear regression and a single decision tree. The `score` function measures a fitted model on the rows it learned from and on the test rows, and adds a line to the results table.

In [ ]:
results = []

def score(name, model, X_rows, y_rows):
    """Score a fitted model on the rows it learned from and on the test rows."""
    pred_train = model.predict(X_rows)
    pred_test = model.predict(X_test)
    results.append({
        'model': name,
        'train R^2': r2_score(y_rows, pred_train),
        'test R^2': r2_score(y_test, pred_test),
        'test RMSE': np.sqrt(mean_squared_error(y_test, pred_test)),
        'test MAE': mean_absolute_error(y_test, pred_test),
    })
    return pd.DataFrame(results).round(3)   # the results table so far

# Linear regression
linear = LinearRegression()
linear.fit(X_train, y_train)
score('Linear regression', linear, X_train, y_train)

# One decision tree, limited to 3 levels with at least 5 rows in each leaf
tree = DecisionTreeRegressor(random_state=42, max_depth=?, min_samples_leaf=?)
tree.fit(X_train, y_train)
score('Single tree (depth 3)', tree, X_train, y_train)

**Question 1.** The RMSE is in dollars. How far off is each baseline on a typical diamond? Which baseline is better?

*Your answer:*

### Step 4: A first XGBoost model

Fit `XGBRegressor` with its default settings: 100 trees, a learning rate of 0.3, and a depth of 6.

In [ ]:
xgb_default = XGBRegressor(
    n_estimators=?,     # number of trees
    learning_rate=?,    # fraction of each tree's correction that is added
    max_depth=?,          # levels of splits allowed in each tree
    random_state=42
)
xgb_default.fit(X_train, y_train)

score('XGBoost (default settings)', xgb_default, X_train, y_train)

**Question 2.** How much did the test RMSE fall compared with the single tree? Compare the training R² with the test R². Is this model overfit as badly as the default model in the lesson (1.0 against 0.737)? Why might a dataset with 53,940 rows behave differently from one with 392?

*Your answer:*

### Step 5: Watch it learn

Fit the same settings on the `X_fit` rows, and ask XGBoost to score itself after every tree on both the rows it learned from and the validation rows.

In [ ]:
xgb_watch = XGBRegressor(n_estimators=100, learning_rate=0.3, max_depth=6, random_state=42)
xgb_watch.fit(
    X_fit, y_fit,
    eval_set=[(X_fit, y_fit), (?, ?)],
    verbose=False
)

# validation_0 is the first eval_set (the fit rows) and validation_1 is the second (the validation rows)
history = xgb_watch.evals_result()
curve = pd.DataFrame({
    'trees': range(1, 101),
    'rows it learned from': history['validation_0']['rmse'],
    'validation rows': history['validation_1']['rmse'],
})

curve.plot(x='trees', figsize=(8, 4.5), linewidth=2)
plt.ylabel('RMSE (dollars)')
plt.title('Error after each tree, default settings')
plt.show()

best = curve['validation rows'].idxmin()
print("Lowest validation RMSE:", round(curve['validation rows'][best], 1), "after", curve['trees'][best], "trees")
print("Validation RMSE after 100 trees:", round(curve['validation rows'].iloc[-1], 1))
print("Training RMSE after 100 trees:", round(curve['rows it learned from'].iloc[-1], 1))

**Question 3.** Describe the two lines. After how many trees does the validation line stop improving? What is happening to the training line after that point?

*Your answer:*

### Step 6: Smaller steps and early stopping

Lower the learning rate to 0.1 and allow up to 2,000 trees. Use early stopping to end training when the validation rows have not improved for 20 trees in a row.

In [ ]:
xgb_slow = XGBRegressor(
    n_estimators=2000,                 # an upper limit; early stopping picks the real number
    learning_rate=?,             # smaller steps
    max_depth=6,
    early_stopping_rounds=?,      # stop when 20 trees in a row fail to improve
    random_state=42
)
xgb_slow.fit(
    X_fit, y_fit,
    eval_set=[(X_val, y_val)],
    verbose=False
)

best_trees = xgb_slow.best_iteration + 1   # best_iteration counts from 0

print("Trees built before stopping:", len(xgb_slow.evals_result()['validation_0']['rmse']))
print("Best number of trees:", best_trees)
print("Validation RMSE at that point:", round(xgb_slow.best_score, 1))

**Question 4.** How many trees were built, and how many were kept? Why are those two numbers different?

*Your answer:*

### Step 7: Compare tree depths

Use the validation rows to compare four tree depths. Each model uses early stopping, so each one chooses its own number of trees. Do not look at the test rows yet.

In [ ]:
tried = []
for depth in [?, ?, ?, ?]:
    model = XGBRegressor(
        n_estimators=2000, learning_rate=0.1, max_depth=depth,
        early_stopping_rounds=20, random_state=42
    )
    model.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=False)
    tried.append({
        'max_depth': depth,
        'trees kept': model.best_iteration + 1,
        'validation RMSE': model.best_score,
    })

tried = pd.DataFrame(tried).round(1)
tried

**Question 5.** Which depth has the lowest validation RMSE? How does the number of trees kept change as the trees get deeper? Why did we use the validation rows for this comparison instead of the test rows?

*Your answer:*

### Step 8: Fit the final model and score it on the test rows

Take the depth with the lowest validation RMSE and the number of trees it kept. Fit a model with those settings on all of the training rows, then score it.

In [ ]:
best_row = tried.sort_values('validation RMSE').iloc[0]
best_depth = int(best_row['max_depth'])
best_trees = int(best_row['trees kept'])
print("Best depth:", best_depth, " Trees:", best_trees)

xgb_final = XGBRegressor(n_estimators=?, learning_rate=0.1, max_depth=?, random_state=42)
xgb_final.fit(?, ?)

summary = score('XGBoost (tuned with validation rows)', xgb_final, X_train, y_train)

plt.figure(figsize=(8, 3))
sns.barplot(data=summary, y='model', x='test RMSE', color='steelblue')
plt.xlabel('RMSE on the test rows (dollars)')
plt.ylabel('')
plt.show()

summary

**Question 6.** A jeweler asks how far off the price estimates will be. Answer in one or two sentences using the test RMSE or MAE of your final model, and compare it with the single tree. How much did tuning add beyond the default XGBoost settings?

*Your answer:*

### Step 9: Read the model

In [ ]:
# Feature importance: each feature's share of the total error reduction across all trees
importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': ?
}).sort_values('importance', ascending=False)

importance.head(8).round(3)

In [ ]:
# Compare actual and predicted prices for the first ten test diamonds
check = df.loc[X_test.index[:10], ['carat', 'cut', 'color', 'clarity', 'price']].copy()
check['predicted'] = xgb_final.predict(X_test.iloc[:10]).round(0)
check['error'] = check['price'] - check['predicted']
check

**Question 7.** Which features matter most? The columns `carat`, `x`, `y` and `z` all measure the size of the diamond. What does that tell you about how to read the importance of any one of them? Can you tell from this table whether a larger diamond costs more or less?

*Your answer:*

### Step 10: A reality check on small data

Go back to the affairs data from the Decision Trees assignment. Copy `affairs.csv` into the same folder as this notebook. It has 601 rows, and the outcome, `has_affair`, is hard to predict.

Fit three classifiers on the same rows: the single decision tree from that assignment, XGBoost with its default settings, and a very cautious XGBoost with 50 trees, a learning rate of 0.05, and a depth of 1.

In [ ]:
affairs = pd.read_csv('?.csv')

numeric_features = ['yearsmarried', 'religiousness_5_is_very_and_1_is_anti',
                    'happiness_in_married_5_is_v_happy_and_1_is_v_unhappy', 'age_decade', 'years_of_education']
categorical_features = ['gender', 'children']

Xa = pd.get_dummies(affairs[numeric_features + categorical_features], columns=categorical_features, drop_first=False, dtype=int)
ya = affairs['?']

Xa_train, Xa_test, ya_train, ya_test = train_test_split(Xa, ya, test_size=0.25, random_state=42)

# The outcome is already 0/1, so no LabelEncoder is needed
models = {
    'Always predict no affair': None,
    'Single tree (depth 3)': DecisionTreeClassifier(random_state=42, max_depth=3, min_samples_leaf=5),
    'XGBoost (default settings)': XGBClassifier(random_state=42),
    'XGBoost (50 trees, rate 0.05, depth 1)': XGBClassifier(n_estimators=?, learning_rate=?, max_depth=?, random_state=42),
}

rows = []
for name, model in models.items():
    if model is None:
        # The simplest possible model: predict the most common class for everyone
        rows.append({'model': name, 'train accuracy': 1 - ya_train.mean(), 'test accuracy': 1 - ya_test.mean()})
    else:
        model.fit(Xa_train, ya_train)
        rows.append({
            'model': name,
            'train accuracy': accuracy_score(ya_train, model.predict(Xa_train)),
            'test accuracy': accuracy_score(ya_test, model.predict(Xa_test)),
        })

pd.DataFrame(rows).round(3)

**Question 8.** Which model has the highest training accuracy? Which has the highest test accuracy? Does any model clearly beat "always predict no affair"? What does this tell you about when XGBoost is worth using?

*Your answer:*